In [7]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# --- کلاس پایه ---
class BaseAnalyzer(object):
    def __init__(self, file_path, output_path, target_sensors):
        self.file_path = file_path
        self.output_path = output_path
        self.target_sensors = target_sensors
        self.df = None
        self.df_model = None

    def load_data(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"❌ فایل یافت نشد: {self.file_path}")
        self.df = pd.read_excel(self.file_path)
        if 'date' in self.df.columns:
            self.df['date'] = pd.to_datetime(self.df['date'])
        self.df_model = self.df.dropna(subset=self.target_sensors).copy()

    def preprocess(self):
        cols = []
        for sensor in self.target_sensors:
            name = f'{sensor}_smooth'
            self.df_model[name] = self.df_model[sensor].rolling(window=5, center=True).mean()
            cols.append(name)
        self.df_model = self.df_model.dropna(subset=cols).copy()
        return cols

    def save_output(self, final_df):
        try:
            os.makedirs(os.path.dirname(self.output_path), exist_ok=True)
            final_df.to_excel(self.output_path, index=False)
            print(f"✅ تعداد {len(final_df)} داده مشکوک برای بازبینی متخصص در مسیر زیر ذخیره شد:\n{self.output_path}")
        except Exception as e:
            print(f"❌ خطا در ذخیره: {e}")

# --- کلاس استخراج داده برای متخصص ---
class SmartBearingAnalyzer(BaseAnalyzer):
    def __init__(self, file_path, output_path, target_sensors, contamination=0.1):
        super().__init__(file_path, output_path, target_sensors)
        self.contamination = contamination

    def run_analysis(self):
        self.load_data()
        smooth_cols = self.preprocess()

        # ۱. استانداردسازی
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(self.df_model[smooth_cols])

        # ۲. مدل‌سازی (پیدا کردن ناهنجاری‌ها)
        model = IsolationForest(n_estimators=100, contamination=self.contamination, random_state=42)
        self.df_model['Behavior_Cluster'] = model.fit_predict(scaled_data)

        # ۳. محاسبه شاخص تخریب (هرچه بالاتر، مشکوک‌تر)
        scores = model.decision_function(scaled_data)
        min_s, max_s = scores.min(), scores.max()
        if max_s - min_s == 0:
            self.df_model['Degradation_Index'] = 0
        else:
            # معکوس‌سازی نمره: ۱ یعنی بدترین وضعیت، ۰ یعنی کاملاً سالم
            self.df_model['Degradation_Index'] = 1.0 - (scores - min_s) / (max_s - min_s)

        # ۴. جداسازی ۱۰ درصدِ بدترین داده‌ها (ناهنجاری‌ها)
        # داده‌هایی که توسط مدل ایزوله شده‌اند (Behavior_Cluster == -1)
        expert_data = self.df_model[self.df_model['Behavior_Cluster'] == -1].copy()

        # مرتب‌سازی بر اساس شاخص تخریب (بحرانی‌ترین‌ها در بالا)

        expert_data = expert_data.sort_values(by='Degradation_Index', ascending=False)

        # ۵. اضافه کردن ستون خالی برای متخصص جهت لیبل‌گذاری
        expert_data['Expert_Label'] = "" # متخصص اینجا 0 یا 1 می‌گذارد
        expert_data['Expert_Comments'] = ""

        self.save_output(expert_data)

# --- تنظیمات و اجرا ---
FILE_INPUT = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
FILE_OUTPUT = r'outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output1.xlsx'
SENSOR_LIST = ['AssetID_9368','AssetID_9369','AssetID_9370','AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# اجرا برای استخراج ۱۰ درصد داده بحرانی
analyzer = SmartBearingAnalyzer(FILE_INPUT, FILE_OUTPUT, SENSOR_LIST, contamination=0.1)
analyzer.run_analysis()

✅ تعداد 588 داده مشکوک برای بازبینی متخصص در مسیر زیر ذخیره شد:
outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output1.xlsx
